# Hypothesis 06 — Regime-Dependent Effects of Sentiment on Returns

## Objective
This notebook investigates whether extreme sentiment acts as a conditional modifier of returns in a regime dependent manner, rather than as a standalone predictive signal.

Building on prior findings, we test the hypothesis that:
- Sentiment does not independently predict returns
- Instead, its effect depends on market structure (regime) and recent price behavior (momentum)

---

## Key Hypotheses

### H1 — Interaction Effect
Extreme sentiment modifies the relationship between momentum and next day returns.

### H2 — Regime Dependence
The effect of extreme sentiment differs between:
- **Mean-reverting regimes**
- **Momentum-driven regimes**

---

## Methodology

### Time Alignment (per metrics.md)
- Sentiment is measured at **t**
- Returns are measured at **t+1**
- Momentum is defined using **t-1 -> t**

This ensures a causal structure without look ahead bias

---

### Feature Construction

- **Momentum**: prior-day return (proxy for trend)
- **Sentiment extremes**:
  - Bottom 10% → `sent_low`
  - Top 10% → `sent_high`
- **Regime**:
  - Momentum regime -> positive prior return
  - Mean-reversion regime -> negative prior return

---

### Modeling Approach

We use Ordinary Least Squares (OLS) with HC1 robust standard errors to estimate conditional relationships.

Models include:
- Baseline (momentum only)
- Sentiment as a standalone factor
- Momentum × sentiment interactions
- Regime dependent interactions

---

## Interpretation Framework

- Coefficients represent conditional associations, not causal effects
- Interaction terms indicate whether:
  - Sentiment modifies momentum
  - Sentiment behaves differently across market regimes

---

## Goal

The goal is to determine whether sentiment:
- Acts as a useful modifier of return behavior
- Is regime dependent
- Can be integrated into a combined signal framework alongside momentum

# 0-Setup

In [2]:
# ----------------------------------------------------
# 0. Setup
# ----------------------------------------------------

import pandas as pd
import numpy as np
import statsmodels.api as sm

FILE_PATH = "../structural/correlation_summary.csv"

df = pd.read_csv(FILE_PATH)

# Ensure proper ordering
df = df.sort_values("price_date").reset_index(drop=True)

# Confirm key columns exist
required_cols = ["price_date", "sentiment_score", "ret_mean"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Ensure datetime
df["price_date"] = pd.to_datetime(df["price_date"], utc=True)

# Basic sanity check
print("Rows:", len(df))
print("Date range:", df["price_date"].min(), "->", df["price_date"].max())

# ----------------------------------------------------
# Time alignment (from metrics.md)
# sentiment: t
# returns:   t+1
# ----------------------------------------------------
print("\nTime alignment check:")
print("sentiment (t) aligned with ret_mean (t+1) after shifting (see next section)")

Rows: 120
Date range: 2024-02-09 00:00:00+00:00 -> 2026-02-23 00:00:00+00:00

Time alignment check:
sentiment (t) aligned with ret_mean (t+1) after shifting (see next section)


# 1-Feature Engineering 

In [4]:
# ----------------------------------------------------
# 1. Feature Engineering
# ----------------------------------------------------

feat_df = df[["price_date", "sentiment_score", "ret_mean"]].copy()

# -------------------------------
# 1. Momentum (t-1 → t)
# -------------------------------
feat_df["momentum"] = feat_df["ret_mean"].shift(1)

# Direction (for interpretation)
feat_df["mom_dir"] = np.where(feat_df["momentum"] > 0, "Up", "Down")

# -------------------------------
# 2. Target (t+1)
# -------------------------------
feat_df["ret_t1"] = feat_df["ret_mean"].shift(-1)

# -------------------------------
# 3. Sentiment extremes (t)
# -------------------------------
q = 0.10

low_cut = feat_df["sentiment_score"].quantile(q)
high_cut = feat_df["sentiment_score"].quantile(1 - q)

feat_df["sent_low"] = (feat_df["sentiment_score"] <= low_cut).astype(int)
feat_df["sent_high"] = (feat_df["sentiment_score"] >= high_cut).astype(int)

# Optional label (for readability/debugging)
feat_df["sent_group"] = "Neutral"
feat_df.loc[feat_df["sent_low"] == 1, "sent_group"] = "Low"
feat_df.loc[feat_df["sent_high"] == 1, "sent_group"] = "High"

# -------------------------------
# 4. Regime definition (simple proxy)
# -------------------------------
# 1 = momentum regime (trend-following)
# 0 = mean-reversion regime

feat_df["regime"] = (feat_df["momentum"] > 0).astype(int)

# -------------------------------
# 5. Rolling context features
# -------------------------------
# These approximate "state" of market instead of single-day signals

# Rolling sentiment (trend in sentiment)
feat_df["sent_roll_5"] = feat_df["sentiment_score"].rolling(5).mean()
feat_df["sent_roll_10"] = feat_df["sentiment_score"].rolling(10).mean()

# Rolling momentum (trend strength)
feat_df["mom_roll_5"] = feat_df["ret_mean"].rolling(5).mean()
feat_df["mom_roll_10"] = feat_df["ret_mean"].rolling(10).mean()

# Rolling volatility (regime intensity / dispersion proxy)
feat_df["vol_roll_5"] = feat_df["ret_mean"].rolling(5).std()

# -------------------------------
# 6. Drop invalid rows
# -------------------------------
feat_df = feat_df.dropna(subset=[
    "momentum",
    "ret_t1",
    "sent_roll_5",
    "mom_roll_5",
    "vol_roll_5"
]).copy()

# -------------------------------
# 7. Final sanity checks
# -------------------------------
print("\nFeature summary:")
print(feat_df[[
    "sent_low", "sent_high", "regime"
]].mean())

print("\nCounts:")
print("Total rows:", len(feat_df))
print("Low sentiment count:", feat_df["sent_low"].sum())
print("High sentiment count:", feat_df["sent_high"].sum())

# Rolling feature overview
print("\nRolling feature preview:")
print(feat_df[[
    "sent_roll_5", "mom_roll_5", "vol_roll_5"
]].describe())

# Preview dataset
print("\nPreview:")
print(feat_df.head())


Feature summary:
sent_low     0.104348
sent_high    0.104348
regime       0.504348
dtype: float64

Counts:
Total rows: 115
Low sentiment count: 12
High sentiment count: 12

Rolling feature preview:
       sent_roll_5  mom_roll_5  vol_roll_5
count   115.000000  115.000000  115.000000
mean     -0.039443    0.001902    0.013826
std       0.058470    0.007691    0.012369
min      -0.146010   -0.016418    0.003733
25%      -0.079111   -0.002112    0.007996
50%      -0.051352    0.001055    0.010606
75%      -0.007882    0.004233    0.014922
max       0.114327    0.031135    0.068023

Preview:
                 price_date  sentiment_score  ret_mean  momentum mom_dir  \
4 2024-03-19 00:00:00+00:00        -0.013564  0.008883 -0.003249    Down   
5 2024-03-25 00:00:00+00:00        -0.119189  0.005030  0.008883      Up   
6 2024-04-04 00:00:00+00:00        -0.182308 -0.017253  0.005030      Up   
7 2024-04-08 00:00:00+00:00         0.019601 -0.002217 -0.017253    Down   
8 2024-04-09 00:00:00+00

# 2-Baseline OLS

In [ ]:
# Baseline model:
# ret_t1 ~ momentum

model_df = feat_df.copy()

X = model_df[["momentum"]]
X = sm.add_constant(X)

y = model_df["ret_t1"]

baseline_ols = sm.OLS(y, X).fit(cov_type="HC1")

print("=== Baseline OLS: ret_t1 ~ momentum ===\n")
print(baseline_ols.summary())

# Quick coefficient view
baseline_coef = pd.DataFrame({
    "coef": baseline_ols.params,
    "std_err": baseline_ols.bse,
    "t_stat": baseline_ols.tvalues,
    "p_value": baseline_ols.pvalues
})

print("\n=== Baseline coefficient table ===\n")
print(baseline_coef)

=== Baseline OLS: ret_t1 ~ momentum ===

                            OLS Regression Results                            
Dep. Variable:                 ret_t1   R-squared:                       0.015
Model:                            OLS   Adj. R-squared:                  0.006
Method:                 Least Squares   F-statistic:                     1.411
Date:                Thu, 02 Apr 2026   Prob (F-statistic):              0.237
Time:                        21:10:19   Log-Likelihood:                 297.80
No. Observations:                 115   AIC:                            -591.6
Df Residuals:                     113   BIC:                            -586.1
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const      

# 3-Add Sentiment

In [6]:
# ----------------------------------------------------
# 3. Add Sentiment (Main Effects)
# ----------------------------------------------------

# Model:
# ret_t1 ~ momentum + sent_low + sent_high

model_df = feat_df.copy()

X = model_df[["momentum", "sent_low", "sent_high"]]
X = sm.add_constant(X)

y = model_df["ret_t1"]

sentiment_ols = sm.OLS(y, X).fit(cov_type="HC1")

print("=== OLS: ret_t1 ~ momentum + sentiment ===\n")
print(sentiment_ols.summary())

# Clean coefficient table
sentiment_coef = pd.DataFrame({
    "coef": sentiment_ols.params,
    "std_err": sentiment_ols.bse,
    "t_stat": sentiment_ols.tvalues,
    "p_value": sentiment_ols.pvalues
})

print("\n=== Coefficient table ===\n")
print(sentiment_coef)

=== OLS: ret_t1 ~ momentum + sentiment ===

                            OLS Regression Results                            
Dep. Variable:                 ret_t1   R-squared:                       0.106
Model:                            OLS   Adj. R-squared:                  0.082
Method:                 Least Squares   F-statistic:                     2.804
Date:                Fri, 03 Apr 2026   Prob (F-statistic):             0.0431
Time:                        20:09:21   Log-Likelihood:                 303.39
No. Observations:                 115   AIC:                            -598.8
Df Residuals:                     111   BIC:                            -587.8
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const   

# 4-Interaction Model

In [7]:
# ----------------------------------------------------
# 4. Interaction Model
# ----------------------------------------------------

# Model:
# ret_t1 ~ momentum + sent_low + sent_high
#         + momentum × sent_low
#         + momentum × sent_high

model_df = feat_df.copy()

# Interaction terms
model_df["mom_x_low"] = model_df["momentum"] * model_df["sent_low"]
model_df["mom_x_high"] = model_df["momentum"] * model_df["sent_high"]

X = model_df[
    ["momentum", "sent_low", "sent_high", "mom_x_low", "mom_x_high"]
]
X = sm.add_constant(X)

y = model_df["ret_t1"]

interaction_ols = sm.OLS(y, X).fit(cov_type="HC1")

print("=== OLS: ret_t1 ~ momentum + sentiment + interactions ===\n")
print(interaction_ols.summary())

interaction_coef = pd.DataFrame({
    "coef": interaction_ols.params,
    "std_err": interaction_ols.bse,
    "t_stat": interaction_ols.tvalues,
    "p_value": interaction_ols.pvalues
})

print("\n=== Interaction coefficient table ===\n")
print(interaction_coef)

=== OLS: ret_t1 ~ momentum + sentiment + interactions ===

                            OLS Regression Results                            
Dep. Variable:                 ret_t1   R-squared:                       0.135
Model:                            OLS   Adj. R-squared:                  0.095
Method:                 Least Squares   F-statistic:                     4.514
Date:                Fri, 03 Apr 2026   Prob (F-statistic):           0.000888
Time:                        20:19:52   Log-Likelihood:                 305.24
No. Observations:                 115   AIC:                            -598.5
Df Residuals:                     109   BIC:                            -582.0
Df Model:                           5                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------

# 5-Interpreting Interaction Effects

In [ ]:
b = interaction_ols.params

# Define scenarios using a representative momentum value
# Use mean absolute momentum for scaling
m_val = model_df["momentum"].abs().mean()

scenarios = []

def compute_ret(momentum, sent_low, sent_high):
    return (
        b["const"]
        + b["momentum"] * momentum
        + b["sent_low"] * sent_low
        + b["sent_high"] * sent_high
        + b["mom_x_low"] * momentum * sent_low
        + b["mom_x_high"] * momentum * sent_high
    )

# Cases
cases = [
    ("Neutral", 0, 0),
    ("Low", 1, 0),
    ("High", 0, 1),
]

for label, low, high in cases:
    scenarios.append({
        "sentiment": label,
        "ret_t1 (momentum +)": compute_ret(m_val, low, high),
        "ret_t1 (momentum -)": compute_ret(-m_val, low, high),
    })

scenario_df = pd.DataFrame(scenarios)

print("\n=== Scenario-based interpretation ===\n")
print(scenario_df)


=== Scenario-based interpretation ===

  sentiment  ret_t1 (momentum +)  ret_t1 (momentum -)
0   Neutral             0.000117             0.001419
1       Low             0.012117             0.022839
2      High            -0.001507            -0.012573


## Interpretation of Interaction Effects

To better understand the interaction model, we evaluate predicted next day returns under different combinations of momentum and sentiment.

### Key Observations

- Under neutral sentiment, momentum exhibits weak mean reverting behavior  
- Under high sentiment, the effect of momentum becomes strongly positive, indicating a shift toward trend following  
- High sentiment also retains a negative baseline effect, suggesting simultaneous overpricing and increased trend persistence  
- Under low sentiment, returns are generally higher, but momentum behavior is not significantly altered  

### Implication

These results indicate that extreme positive sentiment acts as a regime switching variable, changing how the market responds to prior returns. In contrast, negative sentiment primarily introduces a directional bias without altering underlying market dynamics.

## Summary of Findings

- Momentum alone is weak and does not reliably explain next day returns  
- Extreme sentiment improves explanatory power, particularly through high sentiment, which is consistently associated with negative returns  
- Interaction analysis reveals that sentiment is not a standalone signal, but a conditional modifier
- Specifically:
  - High sentiment alters market behavior, shifting it toward momentum driven dynamics  
  - Low sentiment provides a weaker but consistent positive bias  
- These findings support a regime dependent interpretation, where sentiment effectiveness depends on the underlying market state  

### Key Conclusion

Sentiment does not independently predict returns, but instead modifies return behavior in a context dependent manner, with extreme positive sentiment acting as a structural driver of regime shifts.